# CosmUFR Run 4 — quickstart

Five-line inference walkthrough.

Prerequisites: `pip install -e .` from the repo root, and either a local checkpoint at `_local_ckpt/best.pt` or `HF_TOKEN` set in the environment so the loader can pull from `arajgor1/cosmufr-run4`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import cosmufr

# Local first (created by tests/test_inference.py), HF fallback.
ckpt_local = Path("../_local_ckpt/best.pt")
model = cosmufr.load_model(ckpt_path=str(ckpt_local) if ckpt_local.exists() else None)

n = sum(p.numel() for p in model.parameters())
print(f"Loaded CosmUFR Run 4 — {n/1e6:.1f}M parameters")

In [ ]:
# Load a synthetic LCDM-shaped P(k) sample on the standard 200-bin grid.
arr = np.load("synthetic_pk.npy")
pk_z0, pk_z047 = arr[0], arr[1]

result = cosmufr.infer(pk_z0, pk_z047, model=model)

print("Cosmology parameters:")
for k, v in result.params.items():
    s = result.sigmas[k]
    print(f"  {k:>3s} = {v:8.4f}  +/-  {s:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
k = np.exp(result.log_k)
ax[0].loglog(k, pk_z0,    label="input  z=0")
ax[0].loglog(k, 10**result.pk_recon, ls="--", label="reconstructed (z=0)")
ax[0].set_xlabel("k [h/Mpc]")
ax[0].set_ylabel("P(k)")
ax[0].set_title("Input vs reconstructed P(k)")
ax[0].legend()

ax[1].plot(result.energy_log, marker="o")
ax[1].set_xlabel("settling step")
ax[1].set_ylabel("E(b)  (mean over batch)")
ax[1].set_title("Energy descent during settling (16 steps)")
fig.tight_layout()